# Inspec exploration and evaluation

Termfolio, NLP capstone Project 4, Group 9.
Data from the Inspec Database kindly supplied by The IET. [Hulth, 2003](https://aclanthology.org/W03-1028/)

Run `python train.py` from the project root to rebuild the model and the evaluation artifact used here.

In [1]:
import json
from pathlib import Path

root = Path.cwd()
if not (root / 'artifacts').is_dir():
    root = root.parent
evaluation = json.loads((root / 'artifacts/evaluation.json').read_text(encoding='utf-8'))
eda = evaluation['eda']
print(f"{'Split':<12}{'Kept':>6}{'Missing':>9}{'Duplicate':>11}{'Median tokens':>16}{'Gold coverage':>16}")
for split, entry in eda['splits'].items():
    print(f"{split:<12}{entry['documents']:>6}{entry['missing']:>9}{entry['duplicate']:>11}"
          f"{entry['median_words']:>16g}{entry['candidate_coverage']:>15.1%}")

Split         Kept  Missing  Duplicate   Median tokens   Gold coverage
train         1000        0          0             124          72.0%
validation     500        0          0           114.5          70.5%
test           500        0          0             122          72.4%


## Document and reference lengths

The bars are scaled within each distribution.
A four-word cap fits most reference lengths, but some references are absent from the source or excluded by candidate filters.

In [2]:
def histogram(title, entries):
    largest = max(entries.values())
    print(title)
    for label, value in entries.items():
        bar = '█' * max(1, round(value / largest * 32))
        print(f'{label:>10}  {bar:<32}  {value:>5,}')

document_bins = {f'{50*i}–{50*i+49}' if i < 5 else '250+': eda['document_lengths'][str(i)] for i in range(6)}
reference_labels = ('1 word', '2 words', '3 words', '4 words', '5+ words')
reference_bins = dict(zip(reference_labels, (eda['reference_lengths'][str(i)] for i in range(1, 6))))
histogram('Tokens per document', document_bins)
print()
histogram('Words per reference phrase', reference_bins)

Tokens per document
      0–49  █████████                           190
     50–99  ███████████████████████████         533
   100–149  ████████████████████████████████    642
   150–199  ████████████████████                410
   200–249  ████████                            168
      250+  ███                                  57

Words per reference phrase
    1 word  █████████                         2,643
   2 words  ████████████████████████████████  9,856
   3 words  ████████████████                  4,951
   4 words  ████                              1,357
  5+ words  ██                                  467


## Held-out ranking scores

The rankers share a candidate set.
Each metric averages document-level scores across 500 abstracts, using normalized exact matches against uncontrolled references.

In [3]:
results = evaluation['held_out_test']
print(f"{'Method':<18}{'k':>4}{'Precision':>12}{'Recall':>10}{'F1':>10}")
for k in ('5', '10'):
    for method, label in (('learned', 'Learned ranker'), ('tfidf', 'TF-IDF'), ('rake', 'RAKE')):
        score = results[method][k]
        print(f"{label:<18}{k:>4}{score['precision']:>12.3f}{score['recall']:>10.3f}{score['f1']:>10.3f}")

Method               k   Precision    Recall        F1
Learned ranker       5       0.401     0.243     0.284
TF-IDF               5       0.162     0.104     0.119
RAKE                 5       0.140     0.090     0.102
Learned ranker      10       0.320     0.370     0.323
TF-IDF              10       0.139     0.178     0.146
RAKE                10       0.128     0.162     0.134
